## EDA

### Методология формирования аналитических витрин

Для решения задач хакатона была разработана архитектура данных, разделенная на два уровня агрегации. Это позволяет корректно учитывать как статические характеристики лотов, так и динамику пользовательского интереса.

#### 1. Статическая витрина (Факторы ликвидности)

Целевая переменная для каждого объявления $i$ (суммарная ликвидность) определяется как интегральный показатель контактов за весь период экспозиции:

$$L_{i} = \sum_{t=1}^{T_{i}} C_{i, t}$$

Где:
- $L_{i}$ — совокупная ликвидность лота;
- $C_{i, t}$ — количество контактов по объявлению в день $t$;
- $T_{i}$ — общее время жизни объявления в системе.

В случае отсутствия лога активности для конкретного $ID$, значение $L_{i}$ принимается равным $0$. Данная витрина используется для выявления ключевых характеристик автомобиля (цена, пробег, регион), влияющих на итоговый спрос.

#### 2. Динамическая витрина (Анализ временных рядов и VAS)

Для поиска оптимального момента подключения платных услуг (VAS) необходимо анализировать форму кривой затухания спроса. Чтобы избежать смещения оценок из-за "пропусков" в данных (дней без контактов), произведено развертывание векторов времени в плотную сетку $\mathbf{T}_{i} = \{1, 2, \dots, T_{i}\}$. 

Функция ежедневной ликвидности $f(t)$ для объявления $i$ формализована следующим образом:

$$f(t) = 
\begin{cases} 
x_{t}, & \text{при наличии записи в логе в день } t \\
0, & \text{в противном случае}
\end{cases}$$

Такой подход позволяет корректно рассчитывать производную функции спроса и определять точку $t_{opt}$, в которой падение органического интереса требует применения инструментов продвижения.

In [ ]:
import pandas as pd
import numpy as np

In [8]:
# Загрузка
base_df = pd.read_csv('data/raw/liquidity_base.csv')
# Принудительный парсинг дат для исключения ошибок в расчетах
base_df['start_time'] = pd.to_datetime(base_df['start_time'], format='mixed', errors='coerce')
base_df['close_time'] = pd.to_datetime(base_df['close_time'], format='mixed', errors='coerce')

# Загрузка логов с авто-определением разделителя и чисткой названий колонок
cnt_df = pd.read_csv('data/raw/liquidity_cnt.csv', sep=None, engine='python')
cnt_df.columns = cnt_df.columns.str.strip()

# ==========================================
# ВИТРИНА 1: СТАТИКА (ТОТАЛЫ)
# ==========================================
# Используем корректное название 'cnt_contacts'
total_contacts = cnt_df.groupby('id', as_index=False)['cnt_contacts'].sum()
total_contacts.rename(columns={'cnt_contacts': 'total_contacts'}, inplace=True)

df_task1_factors = base_df.merge(total_contacts, on='id', how='left')
df_task1_factors['total_contacts'] = df_task1_factors['total_contacts'].fillna(0)

df_task1_factors.to_csv('data/processed/processed_task1_total_liquidity.csv', index=False)

# ==========================================
# ВИТРИНА 2: ПЛОТНАЯ ДИНАМИКА (ВРЕМЕННЫЕ РЯДЫ)
# ==========================================
# Расчет lifetime
max_date = base_df['close_time'].max() 
base_df['close_time_filled'] = base_df['close_time'].fillna(max_date)
base_df['lifetime_days'] = (base_df['close_time_filled'] - base_df['start_time']).dt.days
base_df['lifetime_days'] = np.where(base_df['lifetime_days'] <= 0, 1, base_df['lifetime_days'])
base_df['lifetime_days'] = base_df['lifetime_days'].fillna(1).astype(int)

# Генерация сетки
grid_list = []
for row in base_df[['id', 'lifetime_days']].itertuples(index=False):
    days = np.arange(1, row.lifetime_days + 1)
    grid_list.append(pd.DataFrame({'id': [row.id] * len(days), 'day': days}))

dense_grid = pd.concat(grid_list, ignore_index=True)

# Мерж с исправленным названием столбца 'cnt_contacts'
df_dynamics_full = dense_grid.merge(cnt_df, on=['id', 'day'], how='left')
df_dynamics_full['cnt_contacts'] = df_dynamics_full['cnt_contacts'].fillna(0)

# Финальный джойн с признаками авто
df_task2_dynamics = df_dynamics_full.merge(
    base_df.drop(columns=['close_time_filled', 'lifetime_days']), 
    on='id', 
    how='inner'
)

df_task2_dynamics.to_csv('data/processed/processed_task2_dense_dynamics.csv', index=False)